# 01 Ingest NEMWeb ZIP Files

List enabled AEMO NEMWeb source folders, skip ZIPs already recorded in the manifest, land unseen ZIP files in Lakehouse Files, and append ingestion control records.

## Configure Pipeline Parameters

This cell defines the parameters normally supplied by a Fabric Data Pipeline: run ID, source filter, lookback window, run limit, and dry-run behaviour.

In [ ]:
# Cell purpose: Configure Pipeline Parameters.
from datetime import datetime, timezone
from pathlib import Path
import os
import hashlib
import uuid

# Fabric pipeline parameters. Override these in the pipeline activity where required.
run_id = str(uuid.uuid4())
source_name = ""  # Blank means all enabled sources.
max_zips_per_run = 500
lookback_hours = 6
dry_run = False

raw_root = "Files/nemweb/raw_zip"

print(f"run_id={run_id}")

## Bootstrap Local Project Package

This cell makes the uploaded `nem_fabric` source package importable in Fabric. Upload `src/nem_fabric` to `Files/libs/nem_fabric` before running the notebook in a Pipeline.

In [ ]:
# Cell purpose: Make nem_fabric importable from Lakehouse Files.
import os
import sys

fabric_lib_path = os.getenv("FABRIC_NOTEBOOK_LIB_PATH", "/lakehouse/default/Files/libs")
if fabric_lib_path not in sys.path:
    sys.path.insert(0, fabric_lib_path)

print(f"Python library path ready: {fabric_lib_path}")

## Load Source Configuration

This cell reads `config/sources.yml`, keeps enabled source folders, and optionally filters to one source. It prevents accidental runs when no source matches.

In [ ]:
# Cell purpose: Load Source Configuration.
from nem_fabric.config import load_yaml
from nem_fabric.nemweb_client import (
    filter_zip_links_by_lookback,
    get_zip_bytes,
    list_zip_links,
)

sources_config = load_yaml("config/sources.yml")
sources = [source for source in sources_config["sources"] if source.get("enabled")]
if source_name:
    sources = [source for source in sources if source["name"] == source_name]

if not sources:
    raise ValueError(f"No enabled source matched source_name={source_name!r}")

print("Sources selected:", [source["name"] for source in sources])

## Define Ingestion Helpers

This cell contains small helper functions for table existence checks, manifest idempotency, Lakehouse path construction, and binary ZIP writes.

In [ ]:
# Cell purpose: Define Ingestion Helpers.
def table_exists(table_name: str) -> bool:
    """Return True when a Lakehouse table exists in the current Spark catalogue."""
    return spark.catalog.tableExists(table_name)


def read_existing_manifest() -> set[str]:
    """Read processed ZIP URLs from the manifest for idempotent ingestion."""
    table_name = "nem_raw_zip_manifest"
    if not table_exists(table_name):
        return set()
    return {row.source_url for row in spark.table(table_name).select("source_url").distinct().collect()}


def lakehouse_raw_path(source: str, filename: str, file_dt) -> str:
    """Build a Lakehouse Files path partitioned by source and date."""
    dt = file_dt or datetime.now(timezone.utc)
    return f"{raw_root}/{source}/{dt:%Y/%m/%d}/{filename}"


def write_lakehouse_binary(relative_path: str, content: bytes) -> None:
    """Write binary ZIP bytes to the default Lakehouse Files mount."""
    mounted_path = Path("/lakehouse/default") / relative_path
    mounted_path.parent.mkdir(parents=True, exist_ok=True)
    with mounted_path.open("wb") as file:
        file.write(content)

## Discover, Filter, and Download ZIPs

This cell lists ZIP files from each selected NEMWeb folder, removes files already recorded in the manifest, downloads unseen files, and builds manifest/log rows.

In [ ]:
# Cell purpose: Discover, Filter, and Download ZIPs.
existing_urls = read_existing_manifest()
manifest_rows = []
log_rows = []

for source in sources:
    links = list_zip_links(source["url"])
    links = filter_zip_links_by_lookback(links, lookback_hours=lookback_hours)
    unseen_links = [link for link in links if link.url not in existing_urls]
    unseen_links = sorted(unseen_links, key=lambda item: item.file_datetime or datetime.min.replace(tzinfo=timezone.utc))[:max_zips_per_run]

    print(f"{source['name']}: {len(unseen_links)} unseen ZIP(s)")
    for link in unseen_links:
        status = "dry_run" if dry_run else "downloaded"
        error_message = ""
        checksum = ""
        byte_count = 0
        target_path = lakehouse_raw_path(source["name"], link.filename, link.file_datetime)
        downloaded_at = datetime.now(timezone.utc).isoformat()

        try:
            content = get_zip_bytes(link.url, dry_run=dry_run)
            byte_count = len(content)
            checksum = hashlib.sha256(content).hexdigest() if content else ""
            if not dry_run:
                write_lakehouse_binary(target_path, content)
        except Exception as exc:
            status = "failed"
            error_message = str(exc)[:4000]

        manifest_rows.append({
            "run_id": run_id,
            "source_name": source["name"],
            "source_url": link.url,
            "source_zip_name": link.filename,
            "source_folder_url": source["url"],
            "file_datetime": link.file_datetime.isoformat() if link.file_datetime else "",
            "lakehouse_path": target_path,
            "checksum": checksum,
            "byte_count": byte_count,
            "first_seen_datetime": downloaded_at,
            "downloaded_datetime": downloaded_at if status == "downloaded" else "",
            "parsed_datetime": "",
            "status": status,
            "error_message": error_message,
        })
        log_rows.append({
            "run_id": run_id,
            "source_name": source["name"],
            "source_url": link.url,
            "source_zip_name": link.filename,
            "status": status,
            "checksum": checksum,
            "first_seen_datetime": downloaded_at,
            "downloaded_datetime": downloaded_at if status == "downloaded" else "",
            "parsed_datetime": "",
            "row_count_bronze": 0,
            "row_count_silver": 0,
            "error_message": error_message,
        })

## Persist Ingestion Control Rows

This cell appends the ZIP manifest and ingestion log Delta tables. If no files are new, it exits without writing duplicate control records.

In [ ]:
# Cell purpose: Persist Ingestion Control Rows.
if manifest_rows:
    spark.createDataFrame(manifest_rows).write.format("delta").mode("append").saveAsTable("nem_raw_zip_manifest")
    spark.createDataFrame(log_rows).write.format("delta").mode("append").saveAsTable("nem_ingestion_log")
    display(spark.createDataFrame(manifest_rows))
else:
    print("No new ZIPs to ingest.")